# LBM-Suite2p-Python: Quickstart

LBM-Suite2p-Python is a volumetric wrapper around [Suite2p](https://suite2p.readthedocs.io/en/latest/) optimized for Light Beads Microscopy data.

## Supported Input Formats

| Format | Extension | Description |
|--------|-----------|-------------|
| TIFF | `.tif`, `.tiff` | Planar timeseries (T, Y, X) |
| Zarr | `.zarr` | Chunked array format |
| Suite2p Binary | `.bin` | Pre-converted binary with `ops.npy` |

**Note:** Raw ScanImage TIFFs must first be assembled using [mbo_utilities](https://millerbrainobservatory.github.io/mbo_utilities/assembly.html).

## Installation

```bash
pip install lbm-suite2p-python
# or from source:
pip install git+https://github.com/MillerBrainObservatory/LBM-Suite2p-Python.git
```

In [1]:
%load_ext autoreload
%autoreload 2
!uv pip install ../../..//mbo_utilities

Using Python 3.12.9 environment at: C:\Users\RBO\repos\LBM-Suite2p-Python\.venv
Resolved 153 packages in 435ms
   Building mbo-utilities @ file:///C:/Users/RBO/repos/mbo_utilities
      Built mbo-utilities @ file:///C:/Users/RBO/repos/mbo_utilities
Prepared 1 package in 1.07s
Uninstalled 1 package in 6ms
Installed 1 package in 29ms
 ~ mbo-utilities==2.2.2 (from file:///C:/Users/RBO/repos/mbo_utilities)


In [2]:
from pathlib import Path
import numpy as np
import mbo_utilities as mbo
import lbm_suite2p_python as lsp

## Prepare Input Files

In [3]:
# Path to assembled planar TIFFs

tif = Path(r"D:\example_extraction\tiff")
t_files = [x for x in tif.glob("*.tif*")]
t_files

[WindowsPath('D:/example_extraction/tiff/plane01_stitched.tif'),
 WindowsPath('D:/example_extraction/tiff/plane02_stitched.tif')]

In [4]:
zarr = Path(r"D:\example_extraction\zarr")
z_files = [x for x in zarr.glob("*.zarr*")]
z_files

[WindowsPath('D:/example_extraction/zarr/plane01_stitched.zarr'),
 WindowsPath('D:/example_extraction/zarr/plane02_stitched.zarr'),
 WindowsPath('D:/example_extraction/zarr/plane03_stitched.zarr'),
 WindowsPath('D:/example_extraction/zarr/plane04_stitched.zarr'),
 WindowsPath('D:/example_extraction/zarr/plane05_stitched.zarr'),
 WindowsPath('D:/example_extraction/zarr/plane06_stitched.zarr'),
 WindowsPath('D:/example_extraction/zarr/plane07_stitched.zarr'),
 WindowsPath('D:/example_extraction/zarr/plane08_stitched.zarr'),
 WindowsPath('D:/example_extraction/zarr/plane09_stitched.zarr'),
 WindowsPath('D:/example_extraction/zarr/plane10_stitched.zarr'),
 WindowsPath('D:/example_extraction/zarr/plane11_stitched.zarr'),
 WindowsPath('D:/example_extraction/zarr/plane12_stitched.zarr'),
 WindowsPath('D:/example_extraction/zarr/plane13_stitched.zarr'),
 WindowsPath('D:/example_extraction/zarr/plane14_stitched.zarr')]

In [5]:
# or: use binary already prepared for suite2p
# input_files = mbo.get_files(data_dir, str_contains="data_raw.bin", max_depth=2)

---

## Configure Parameters

The `ops` dictionary controls all Suite2p and Cellpose parameters. You only need to specify values you want to override from defaults.

In [6]:
# For Cellpose anatomical detection (recommended for LBM data)
ops = {
    "diameter": 4,                   # Expected cell diameter (pixels)
}

# Where to save results
save_path = Path(r"D:\example_extraction\suite2p")

---

## Process a Single Plane: `run_plane()`

Use `run_plane()` to process one z-plane. This is useful for:
- Testing parameters before full volume processing
- Processing individual planes of interest

Results are saved to `save_path/{plane_name}/` subdirectory by default.

In [7]:
# View function signature
help(lsp.run_plane)

Help on function run_plane in module lbm_suite2p_python.run_lsp:

run_plane(input_path: str | pathlib.Path, save_path: str | pathlib.Path | None = None, ops: dict | str | pathlib.Path = None, chan2_file: str | pathlib.Path | None = None, keep_raw: bool = False, keep_reg: bool = True, force_reg: bool = False, force_detect: bool = False, dff_window_size: int = 300, dff_percentile: int = 20, save_json: bool = False, plane_name: str | None = None, **kwargs) -> pathlib.Path
    Processes a single imaging plane using suite2p, handling registration, segmentation,
    and plotting of results.

    Parameters
    ----------
    input_path : str or Path
        Full path to the file to process, with the file extension.
    save_path : str or Path, optional
        Root directory to save the results. A subdirectory will be created based on
        the input filename or `plane_name` parameter.
    ops : dict, str or Path, optional
        Path to or dict of user‐supplied ops.npy. If given, it over

In [8]:
print(save_path)

D:\example_extraction\suite2p


In [9]:
# Process a single plane
# Results saved to save_path/plane01_stitched/ (derived from input filename)
ops_file = lsp.run_plane(
    input_path=t_files[0],           # Input file
    save_path=save_path,             # Root output directory
    ops=ops,                         # Parameter overrides
    keep_raw=False,                  # Delete raw binary to save space
    keep_reg=True,                   # Keep registered binary for GUI viewing
    force_reg=False,                 # Skip registration if already done
    force_detect=False,              # Skip detection if stat.npy exists
    # plane_name="custom_name",      # Optional: override subdirectory name
)

Importing suite2p packages...
Roi detection skipped, stat.npy already exists for plane 1.
Skipping data_raw.bin write, already exists and passes data validation checks.
Registration skipped - copying data_raw.bin to data.bin...
NOTE: applying default C:\Users\RBO\.suite2p\classifiers\classifier_user.npy
Saved panel TIFF to D:\example_extraction\suite2p\plane01_stitched\pc_metrics_panels.tif
Saved metrics CSV to D:\example_extraction\suite2p\plane01_stitched\pc_metrics.csv
   Rigid    Avg_NR    Max_NR
0    0.0  0.046224  0.200000
1    0.0  0.044421  0.200000
2    0.0  0.031530  0.141421
3    0.0  0.031939  0.141421
4    0.0  0.039659  0.200000
Plotting results for 317 accepted / 28 rejected ROIs
  Saved: 05_quality_diagnostics.png


---

## 3b. Process Full Volume: `run_volume()`

Use `run_volume()` to process all z-planes and generate volumetric statistics.

In [8]:
# Process entire volume
output_ops = lsp.run_volume(
    input_files=z_files,         # List of all plane files
    save_path=save_path,             # Output directory
    ops=ops,                # Parameter overrides
    keep_raw=False,                  # Delete raw binaries
    keep_reg=True,                   # Keep registered binaries
    force_reg=False,                 # Skip if already registered
    force_detect=False,              # Skip if already detected
)

print(f"Processed {len(output_ops)} planes")

Importing suite2p packages...
Roi detection skipped, stat.npy already exists for plane 1.
Skipping data_raw.bin write, already exists and passes data validation checks.
Registration skipped - copying data_raw.bin to data.bin...
NOTE: applying default C:\Users\RBO\.suite2p\classifiers\classifier_user.npy
Saved panel TIFF to D:\example_extraction\suite2p\plane01_stitched\pc_metrics_panels.tif
Saved metrics CSV to D:\example_extraction\suite2p\plane01_stitched\pc_metrics.csv
   Rigid    Avg_NR    Max_NR
0    0.0  0.046224  0.200000
1    0.0  0.044421  0.200000
2    0.0  0.031530  0.141421
3    0.0  0.031939  0.141421
4    0.0  0.039659  0.200000
Plotting results for 317 accepted / 28 rejected ROIs
  Saved: 05_quality_diagnostics.png
Completed plane01_stitched.zarr -> D:\example_extraction\suite2p\plane01_stitched\ops.npy
Time for D:\example_extraction\zarr\plane01_stitched.zarr: 0.1 min
CPU 11.6% | RAM 28.52 GB
Importing suite2p packages...
Roi detection skipped, stat.npy already exists f

## Output Files

Processing generates standard Suite2p outputs plus additional visualization files.

In [14]:
from pprint import pprint

# List output files
output_files = sorted(save_path.rglob("*.*"))
print("Output files:")
pprint([f.relative_to(save_path).__str__() for f in output_files[:20]])

Output files:
['all_neurons_5770acc_761rej.png',
 'all_planes_masks.png',
 'correlation_image.png',
 'correlation_image_segmentation.png',
 'data.bin',
 'F.npy',
 'Fneu.npy',
 'iscell.npy',
 'max_projection_image.png',
 'max_projection_segmentation.png',
 'mean_image.png',
 'mean_image_enhanced.png',
 'mean_image_enhanced_segmentation.png',
 'mean_image_segmentation.png',
 'mean_volume_signal.png',
 'model.npy',
 'ops.npy',
 'pc_metrics.csv',
 'pc_metrics_panels.tif',
 'plane01_stitched\\correlation_image.png']


### Planar Outputs

Each z-plane directory contains:

#### Data Files

| File | Shape | Description |
|------|-------|-------------|
| `ops.npy` | dict | Processing parameters and metadata |
| `stat.npy` | (n_rois,) | ROI definitions (pixel coordinates, weights, shape stats) |
| `F.npy` | (n_rois, n_frames) | Raw fluorescence traces |
| `Fneu.npy` | (n_rois, n_frames) | Neuropil fluorescence traces |
| `spks.npy` | (n_rois, n_frames) | Deconvolved spike estimates |
| `iscell.npy` | (n_rois, 2) | Cell classification: `[:, 0]` = is_cell (0/1), `[:, 1]` = probability |
| `data.bin` | (n_frames, Ly, Lx) | Registered movie (if `keep_reg=True`) |

#### Visualization Files

Files are numbered to ensure proper ordering when viewing in file browsers.

| File | Description |
|------|-------------|
| `01_correlation.png` | Pixel-wise correlation image |
| `01_correlation_segmentation.png` | Correlation image with ROI overlay |
| `02_max_projection.png` | Maximum intensity projection |
| `02_max_projection_segmentation.png` | Max projection with ROI overlay |
| `03_mean.png` | Temporal mean image |
| `03_mean_segmentation.png` | Mean image with ROI overlay |
| `04_mean_enhanced.png` | Enhanced mean image (edge sharpened) |
| `04_mean_enhanced_segmentation.png` | Enhanced mean with ROI overlay |
| `05_quality_diagnostics.png` | ROI size, SNR, compactness metrics |
| `06_registration.png` | Registration quality visualization |
| `07_traces_raw.png` | Sample raw fluorescence traces |
| `08_traces_dff.png` | Sample ΔF/F traces |
| `09_traces_noise.png` | Noise estimation traces (rejected ROIs) |
| `10_noise_accepted.png` | Shot noise histogram (accepted) |
| `11_noise_rejected.png` | Shot noise histogram (rejected) |
| `12_rastermap.png` | Activity sorted by similarity |

### Volumetric Outputs

When processing multiple planes with `run_volume()`, additional files are generated in the root save directory:

| File | Description |
|------|-------------|
| `all_planes_masks.png` | Grid showing ROI masks overlaid on mean images for all planes |
| `volume_quality_metrics.png` | Compactness, skewness, ROI size, and radius per plane (mean ± std) |
| `volume_trace_analysis.png` | Example traces, SNR and fluorescence per plane, activity heatmap |
| `volume_summary.csv` | Per-plane statistics table (ROIs, SNR, acceptance rate) |
| `mean_volume_signal.png` | Mean signal intensity across z-depth |
| `rastermap.png` | Activity sorted by similarity across all planes |
| `volume_stats.npy` | Per-plane statistics dictionary |

---

## 5. Load and Analyze Results

In [ ]:
# Find ops files
ops_files = mbo.get_files(save_path, str_contains="ops.npy", max_depth=3)
print(f"Found {len(ops_files)} processed planes")

# Load results from first plane
if ops_files:
    results = lsp.load_planar_results(ops_files[0])

    # iscell is (n_rois, 2): column 0 is classification (0/1), column 1 is probability
    iscell_mask = results['iscell'][:, 0].astype(bool)

    print(f"\nPlane results:")
    print(f"  Total ROIs: {len(results['stat'])}")
    print(f"  Accepted cells: {iscell_mask.sum()}")
    print(f"  Frames: {results['F'].shape[1]}")
    print(f"  F shape: {results['F'].shape}")

### Calculate ΔF/F

In [ ]:
# Calculate ΔF/F with rolling percentile baseline
if ops_files:
    dff = lsp.dff_rolling_percentile(
        results['F'],
        window_size=300,    # frames (~10× tau × fs)
        percentile=20       # baseline percentile
    )

    # Filter for accepted cells only using the classification column
    iscell_mask = results['iscell'][:, 0].astype(bool)
    dff_cells = dff[iscell_mask]
    print(f"ΔF/F shape (accepted cells): {dff_cells.shape}")

---

## 6. Open Suite2p GUI

The Suite2p GUI provides interactive visualization and manual curation:

- **View registered movie**: Suite2p → Registration → View Registration Binary
- **Compare raw vs registered**: Check "View raw binary" (requires `keep_raw=True`)
- **Registration quality**: Suite2p → Registration → View Registration Metrics (>1500 frames)

In [13]:
# Open GUI for manual curation
run_gui = False
if ops_files and run_gui:
    stat_file = ops_files[0].parent / "stat.npy"
    if stat_file.exists():
        from suite2p import gui
        gui.run(statfile=str(stat_file))

---

## 7. Volumetric Analysis

When processing multiple z-planes with `run_volume()`, the pipeline automatically generates publication-quality figures for comprehensive volumetric analysis. You can also generate these manually:

In [15]:
# Load data from all planes for volumetric analysis
all_results = []
for ops_file in ops_files:
    try:
        res = lsp.load_planar_results(ops_file)
        all_results.append(res)
    except Exception as e:
        print(f"Error loading {ops_file}: {e}")

if len(all_results) > 1:
    # Consolidate all planes
    all_stat = np.concatenate([res["stat"] for res in all_results])
    all_iscell = np.vstack([res["iscell"] for res in all_results])
    all_F = np.concatenate([res["F"] for res in all_results], axis=0)
    all_Fneu = np.concatenate([res["Fneu"] for res in all_results], axis=0)

    print(f"Consolidated volume:")
    print(f"  Total ROIs: {len(all_stat)}")
    print(f"  Accepted: {all_iscell[:, 0].sum()}")
    print(f"  Planes: {len(all_results)}")
    print(f"  F shape: {all_F.shape}")

NameError: name 'ops_files' is not defined

### Multi-Plane Mask Overview

Visualize detected ROIs across all planes in a single figure:

In [ ]:
# Plot masks from all planes in a grid
if len(all_results) > 1:
    fig = lsp.plot_multiplane_masks(
        suite2p_path=save_path,
        stat=all_stat,
        iscell=all_iscell,
        nrows=3,
        ncols=5,
        save_path=save_path / "all_planes_masks.png"  # Optional: save to file
    )

### Volume Quality Metrics

Generate publication-quality figures showing ROI quality across all planes:

In [ ]:
# Quality metrics: ROI counts, compactness, size distributions
if len(all_results) > 1:
    fig = lsp.plot_plane_quality_metrics(
        stat=all_stat,
        iscell=all_iscell,
        save_path=save_path / "volume_quality_metrics.png",
        style="publication"  # or "dark" for dark background
    )

### Trace Analysis

Comprehensive analysis of fluorescence traces across the volume:

In [ ]:
# Trace analysis: SNR, correlation, activity heatmap
if len(all_results) > 1:
    first_ops = lsp.load_ops(ops_files[0])

    fig, metrics = lsp.plot_trace_analysis(
        F=all_F,
        Fneu=all_Fneu,
        stat=all_stat,
        iscell=all_iscell,
        ops=first_ops,
        save_path=save_path / "volume_trace_analysis.png"
    )

    # Access computed metrics
    print(f"Mean SNR across volume: {np.mean(metrics['snr']):.2f}")
    print(f"Cells with SNR > 2: {np.sum(metrics['snr'] > 2)} ({100*np.mean(metrics['snr'] > 2):.1f}%)")

### Summary Statistics Table

Generate a comprehensive summary table for all planes:

In [ ]:
# Create summary table with per-plane statistics
if len(all_results) > 1:
    summary_df = lsp.create_volume_summary_table(
        stat=all_stat,
        iscell=all_iscell,
        F=all_F,
        Fneu=all_Fneu,
        ops=first_ops,
        save_path=save_path / "volume_summary.csv"
    )

    # Display the table
    print(summary_df.to_string(index=False))

### Automatic Volumetric Outputs

When using `run_volume()`, these figures are automatically generated in the output directory:

| File | Description |
|------|-------------|
| `all_planes_masks.png` | Grid showing ROIs overlaid on mean images for all planes |
| `volume_quality_metrics.png` | ROI counts, compactness, size, and acceptance rates per plane |
| `volume_trace_analysis.png` | SNR distributions, example traces, correlation matrix, activity heatmap |
| `volume_summary.csv` | Per-plane statistics table (ROIs, SNR, acceptance rate) |
| `mean_volume_signal.png` | Mean signal intensity across z-depth |
| `rastermap.png` | Activity sorted by similarity (requires rastermap package) |

---

## Next Steps

- **Parameter tuning**: See [anatomical_grid_search.ipynb](./anatomical_grid_search.ipynb) for systematic parameter optimization
- **Spike inference**: See [tau_spike_inference_analysis.ipynb](./tau_spike_inference_analysis.ipynb) for tau parameter analysis
- **Volume consolidation**: Use `lsp.consolidate_volume()` to merge results from all planes into a single directory
- **Documentation**: See the [User Guide](https://millerbrainobservatory.github.io/LBM-Suite2p-Python/user_guide.html) for detailed parameter explanations